# Solar Filament V29 — Novelty-Credit Rank Ensemble + Supervised Fragment Candidates

**Approach motivated by public competitor evidence, not another pretrained detector, generic U-Net, or confidence sweep.** The historical V25 public score **0.39** remains the locked control. V29 extracts otherwise rejected candidates from the same validated two-detector system, adds a small number of hypothesis-driven fragment unions, and learns *incremental panoptic-quality credit* conditional on V25's already detected instances. HistGradientBoosting, ExtraTrees, RandomForest and a validity classifier are trained on competition TRAIN annotations only, in calibration date groups. No use of annotations from the 180 test images or external MAGFiLO releases.

The published competitor [Artem Shelamanov](https://www.shelamanov.com/portfolio/solar) reports 0.39 from a 77-feature scorer plus merged fragments and 0.40 from rank-averaged scores; other public approaches also tried U-Net, multiple YOLO seeds and Mask R-CNN without necessarily improving their public PQ. Our novelty is **credit for a missing instance**, not simply a high-IoU mask to an already-detected instance. Use the official organizer PQ notebook to verify scores. All improvements here are *hypotheses until Kaggle scores the CSV*.

**Kaggle attach**: official `solar3k`, plus V25/26 full checkpoint ZIP or extracted V20/V24/V25 weights, plus original `V25_REAL_REFINE_PLUS_QUALITY.csv` (may be embedded in `SOLAR_V25_ALL_PREDICTIONS_AND_REPORTS.zip`). GPU T4/T4×2. No fresh deep-net training required; new models are trained on instance features and expert labels. On a long run, use **Save Version → Save & Run All** and save outputs.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input');OUT=Path('/kaggle/working/solar_v29')
SEED=2026;GPU_ID=0;H=W=2048
RAW_CONF=.08;RAW_IOU=.72;MAX_DET=120
ALLOW_NON_V20_WEIGHT_HASH=False;REQUIRE_V24_MODELS=True;FULL_VALIDATION_REQUIRED=True
V25_EXPECTED_QUALITY_CUTOFF=.38
V25_EXPECTED_CSV_SHA='ac470fbfa0e2ca5f2d'; V25_EXPECTED_ROWS=1093
CROP_SIDE=320
CURVE_WINDOW=768;CURVE_STRIDE=640;CURVE_SIDE=512;CURVE_MIN_AREA=170;CURVE_MAX_PROPOSALS=42
# V29: independently ranked recovery candidates, not retraining a baseline that already works.
LOW_GLOBAL=.115;LOW_TILE=.145;LOW_CURVE=.69;MIN_EXTRA_AREA=180
MAX_GLOBAL_RAW=26;MAX_TILE_RAW=42;MAX_CURVE_RAW=8
MAX_SINGLES_PER_SCENE=28;MAX_FRAGMENT_PAIRS=9
PAIR_END_DISTANCE=90.; PAIR_MAX_IOU=.12
MAX_NEW_ACCEPTED_PER_SCENE=4
MAX_ACCEPTED_OVERLAP=.22; MIN_FREE_FRACTION=.78
CANDIDATE_THRESHOLDS=[.035,.055,.075,.095,.125,.16,.20,.24,.28]
MIN_PROMOTION_AUDIT_GAIN=.004;MIN_BOOTSTRAP_POSITIVE=.90
SCORER_TRAINING_SPLIT='historical_calibration_53_scenes';TEST_LABELS_FORBIDDEN=True
for d in ('models','raw','refined','curve_raw','reports','submissions','input_data','candidate_cache','feature_cache','labels','checkpoints'):
    (OUT/d).mkdir(parents=True,exist_ok=True)
assert INPUT.is_dir(),'Run on Kaggle with solar3k and V25 assets.'
print('V29 output:',OUT)


In [ ]:
import os,sys,time,json,hashlib,zipfile,shutil,gzip,random,subprocess,importlib.util
import gc as _gc_safe
from collections import defaultdict
import numpy as np,pandas as pd,cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from IPython.display import display

def require(module,package):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
require('pycocotools','pycocotools')
# Lock exact V22 version when available to maximize baseline reproduction.
try:
    from importlib.metadata import version as package_version
    installed_version=package_version('ultralytics')
except Exception:
    installed_version=None
if installed_version!='8.4.189':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.189'])
from pycocotools import mask as coco
from ultralytics import YOLO
import ultralytics,torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU T4/T4x2. No CPU or nano fallback.'
assert torch.cuda.device_count()>GPU_ID
print('Using',torch.cuda.get_device_name(GPU_ID),'ultralytics',ultralytics.__version__)
cv2.setNumThreads(0);random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
H=W=2048
EXT={'.jpeg','.jpg','.png','.tif','.tiff','.bmp'}
def files_in(d):
    p=Path(d)
    return sorted(q for q in p.iterdir() if q.is_file() and q.suffix.lower() in EXT) if p.is_dir() else []
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def save_json(p,obj):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_suffix(p.suffix+'.tmp');temp.write_text(json.dumps(obj,indent=2,default=str));temp.replace(p)
def enc(m):return coco.encode(np.asfortranarray(np.asarray(m,np.uint8)))
def txt(r):return r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else str(r['counts'])
def as_rle(x):return {'size':[H,W],'counts':x.encode('ascii') if isinstance(x,str) else x}
assert callable(_gc_safe.collect)
save_json(OUT/'reports'/'env.json',{'torch':torch.__version__,'ultralytics':ultralytics.__version__,'gpu':torch.cuda.get_device_name(GPU_ID),'no_prior_split_required':True,'prior_verified_public_PQ':0.39,'target_public_PQ':0.40})

## 1. Official data and the historical V20 acquisition-day partition

In [ ]:
ANAME='MAGFiLO_1.0_Annotations_kaggle2026_train.json'
found=sorted(INPUT.rglob(ANAME))
OFFICIAL=next((p.parent.parent for p in found if (p.parent.parent/'test'/'test_images').is_dir() and (p.parent.parent/'train'/'train_images').is_dir()),None)

def find_zip(name):
    matches=list(INPUT.rglob(name))
    return matches[0] if matches else None

def safe_images_unzip(archive,destination):
    destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            if not item.filename.lower().endswith(tuple(EXT)):continue
            target=destination/Path(item.filename).name
            if not target.exists():
                with z.open(item) as src,open(target,'wb') as dst:shutil.copyfileobj(src,dst,1<<20)
    return destination
if OFFICIAL:
    ANNOT=OFFICIAL/'train'/ANAME
    TRAIN={p.stem:p for p in files_in(OFFICIAL/'train'/'train_images')}
    TEST={p.stem:p for p in files_in(OFFICIAL/'test'/'test_images')}
    print('Using full official competition folder',OFFICIAL)
else:
    annot=found[0] if found else None
    ztest=find_zip('test.zip');ztrain=find_zip('sometrain.zip')
    assert annot is not None and ztest is not None, 'Attach official solar3k, or the annotation JSON and test.zip as Kaggle inputs.'
    ANNOT=annot
    TEST={p.stem:p for p in files_in(safe_images_unzip(ztest,OUT/'input_data'/'test_images'))}
    TRAIN={p.stem:p for p in files_in(safe_images_unzip(ztrain,OUT/'input_data'/'train_images'))} if ztrain else {}
    print('Using upload ZIP fallback. Validation may cover fewer than 104 physical images.')
assert len(TEST)==180,('Expected 180 test scenes; found',len(TEST))
assert len(set(TEST)&set(TRAIN))==0
DOC=json.loads(ANNOT.read_text())
BY_ID=defaultdict(list)
for ann in DOC['annotations']:BY_ID[str(ann['image_id'])].append(ann)
VERSIONS=defaultdict(list)
for item in DOC['images']:
    stem=Path(item['file_name']).stem
    if stem in TRAIN:VERSIONS[stem].append((str(item['id']),BY_ID.get(str(item['id']),[])))
assert set(TRAIN)==set(VERSIONS)
if TRAIN:
    days=defaultdict(list)
    for img in DOC['images']:
        stem=Path(img['file_name']).stem
        days[stem[:8]].append(stem)
    days={k:sorted(set(v)) for k,v in days.items()}
    shuffled=sorted(days)
    random.Random(SEED+2020).shuffle(shuffled)
    selected_days=[];n=0
    for d in shuffled:
        if n>=104:break
        selected_days.append(d);n+=len(days[d])
    VALIDATION=sorted(set(stem for d in selected_days for stem in days[d]) & set(TRAIN))
else:VALIDATION=[]
FULL_V20_VALIDATION=(len(VALIDATION)==104 and len(TRAIN)==707)
print('Images: train present',len(TRAIN),'test',len(TEST),'V20 tune available',len(VALIDATION),'full reproduction',FULL_V20_VALIDATION)
if VALIDATION:
    assert not ({s[:8] for s in VALIDATION} & {s[:8] for s in TRAIN if s not in VALIDATION})
save_json(OUT/'reports'/'dataset.json',{'annotation_sha256':sha(ANNOT),'train_available':len(TRAIN),
    'test_images':len(TEST),'validation_available':len(VALIDATION),'full_v20_reproduction':FULL_V20_VALIDATION,
    'validation_stems':VALIDATION,'test_ground_truth_used':False})

## 2. Restore verified frozen V20 detectors

In [ ]:
EXPECT={
 'portable_global_m':('portable_global_m_best.pt','73fd3f3a4931'),
 'portable_native_m':('portable_native_m_best.pt','1a932d28e590'),
}
V20_DIR=Path('/kaggle/input/datasets/lighittha/solarouts/solar_v20_from_scratch/models')
ZIPS=sorted(set(p for p in INPUT.rglob('*.zip') if any(token in p.name.lower() for token in ('v20','v21','weights','models'))))
WEIGHTS={};HASHES={};SOURCES={}
(OUT/'models').mkdir(parents=True,exist_ok=True)
for model_name,(filename,prefix) in EXPECT.items():
    path=OUT/'models'/filename
    choices=[]
    direct=V20_DIR/filename
    if direct.is_file():choices.append(direct)
    choices+=[p for p in INPUT.rglob(filename) if p.is_file() and p!=direct]
    if path.is_file():choices.insert(0,path)
    if not path.is_file() or path.stat().st_size<20_000_000:
        restored=False
        for src in choices:
            if 30_000_000<src.stat().st_size<65_000_000:
                if src!=path:shutil.copy2(src,path)
                SOURCES[model_name]=str(src);restored=True;break
        if not restored:
            for archive in ZIPS:
                try:
                    with zipfile.ZipFile(archive) as z:
                        member='models/'+filename
                        if member not in z.namelist():continue
                        if not (30_000_000<z.getinfo(member).file_size<65_000_000):continue
                        temp=path.with_suffix('.part')
                        with z.open(member) as rf,open(temp,'wb') as wf:shutil.copyfileobj(rf,wf,1<<20)
                        temp.replace(path);SOURCES[model_name]=str(archive)+'::'+member
                        restored=True;break
                except (OSError,zipfile.BadZipFile) as e:print('Ignore corrupt ZIP:',archive,e)
        if not restored:raise FileNotFoundError('Missing trained V20 '+filename+f'. Expected at {V20_DIR/filename} or inside the V20 weights ZIP. This notebook never substitutes pretrained-only or nano models.')
    digest=sha(path)
    if not ALLOW_NON_V20_WEIGHT_HASH:
        assert digest.startswith(prefix),f'{filename}: wrong weights hash! Expected V20 prefix {prefix}, got {digest[:12]}'
    m=YOLO(str(path))
    n_params=sum(p.numel() for p in m.model.parameters())
    assert m.task=='segment' and 21_000_000<=n_params<=24_000_000,(model_name,m.task,n_params)
    del m;_gc_safe.collect()
    WEIGHTS[model_name]=path;HASHES[model_name]=digest
print('Recovered V20 model identity:',{k:v[:12] for k,v in HASHES.items()})
save_json(OUT/'reports'/'weights.json',{'sha256':HASHES,'source':SOURCES,'V20_checkpoint_hash_gate':not ALLOW_NON_V20_WEIGHT_HASH})

## 3. Immutable V22 detector proposal generation and original mask selection

In [ ]:
# Locked V22 successful source definitions.
SRC_BASE_GLOBAL=('portable_global_m','g1536')
SRC_BASE_NATIVE=('portable_native_m','g1536')
SRC_HI=('portable_global_m','g1792')
SRC_TILES=('portable_native_m','t1280')
SRC_HIFLIP=('portable_global_m','g1792_hflip')
BASE_SOURCES=[SRC_HI,SRC_TILES]
EXTRA_SOURCES=[]
ACTIVE=BASE_SOURCES+EXTRA_SOURCES
ENGINES={}
def get_engine(name):
    if name not in ENGINES:
        ENGINES.clear();_gc_safe.collect();torch.cuda.empty_cache()
        ENGINES[name]=YOLO(str(WEIGHTS[name]))
    return ENGINES[name]

def view_windows(view):
    if view=='g1536':return [(0,0,2048,1536,False)]
    if view=='g1792':return [(0,0,2048,1792,False)]
    if view=='g1792_hflip':return [(0,0,2048,1792,True)]
    if view=='t1280':return [(x,y,1280,1024,False) for y in (0,768) for x in (0,768)]
    raise ValueError(view)
def sig(source):
    name,view=source
    return hashlib.sha256(json.dumps({'model_sha256':HASHES[name],'view':view,
      'conf':RAW_CONF,'iou':RAW_IOU,'max_det':MAX_DET,'retina_masks':True,
      'agnostic_nms':True,'ultralytics':ultralytics.__version__,'version':'V22_1' if view in ('g1792','t1280','g1536') else 'V24_1'},sort_keys=True).encode()).hexdigest()
def cache_file(source,stem):
    name,view=source
    return OUT/'raw'/name/view/(stem+'.json.gz')
@torch.inference_mode()
def raw_predictions(source,stem):
    path=cache_file(source,stem);identifier=sig(source)
    if path.exists():
        try:
            with gzip.open(path,'rt') as f:v=json.load(f)
            if v.get('signature')==identifier and v.get('stem')==stem:return v['predictions']
        except (OSError,ValueError,KeyError):pass
    image_path=TRAIN.get(stem) or TEST.get(stem)
    if image_path is None:raise FileNotFoundError('Missing scene: '+stem)
    gray=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
    assert gray is not None and gray.shape==(H,W),(stem, None if gray is None else gray.shape)
    predictions=[]
    name,view=source
    model=get_engine(name)
    for x,y,side,net,hflip in view_windows(view):
        roi=np.ascontiguousarray(gray[y:y+side,x:x+side])
        if hflip:roi=cv2.flip(roi,1)
        image=cv2.cvtColor(roi,cv2.COLOR_GRAY2BGR)
        result=model.predict(image,imgsz=net,conf=RAW_CONF,iou=RAW_IOU,max_det=MAX_DET,
                             agnostic_nms=True,retina_masks=True,device=GPU_ID,verbose=False)[0]
        if result.masks is None or result.boxes is None:continue
        mask_data=result.masks.data.detach().cpu().numpy()
        confidences=result.boxes.conf.detach().cpu().numpy()
        for mask_arr,conf in zip(mask_data,confidences):
            mask=(mask_arr>0.5).astype(np.uint8)
            if mask.shape!=(side,side):mask=cv2.resize(mask,(side,side),interpolation=cv2.INTER_NEAREST)
            if hflip:mask=np.ascontiguousarray(cv2.flip(mask,1))
            if int(mask.sum())<100:continue
            if side!=2048:
                ys,xs=np.where(mask)
                truncated=bool((x>0 and xs.min()<10) or (x+side<W and xs.max()>=side-10) or
                               (y>0 and ys.min()<10) or (y+side<H and ys.max()>=side-10))
                full=np.zeros((H,W),np.uint8)
                full[y:y+side,x:x+side]=mask
            else:truncated=False;full=mask
            rr=enc(full)
            predictions.append({'counts':txt(rr),'score':float(conf),'area':int(coco.area(rr)),
                                'truncated':truncated,'source_view':view})
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with gzip.open(tmp,'wt',compresslevel=3) as f:
        json.dump({'signature':identifier,'stem':stem,'view':view,'predictions':predictions},f,separators=(',',':'))
    tmp.replace(path)
    return predictions
print('V23 views including immutable winner:',ACTIVE)

In [ ]:
def recipe(name,sources,gconf=.24,tconf=.32,area=200,seam=False,refine='none',minfree=.45):
    return {'name':name,'sources':[list(s) for s in sources],'global_conf':float(gconf),
            'tile_conf':float(tconf),'min_area':int(area),'seam_guard':bool(seam),
            'nms_iou':.50,'tile_bonus':.08,'min_free':float(minfree),'refine':refine}

# This exact configuration produced V22_ALT_2.csv (public PQ 0.37).
WINNER=recipe('V22_ALT2_WINNER_037',[SRC_HI,SRC_TILES],
              gconf=.24,tconf=.42,area=200,seam=True,refine='none',minfree=.45)
WINNER['nms_iou']=.50
WINNER['tile_bonus']=.08
WINNER['engine']='v22'
WINNER['fusion']='none'
WINNER['tile_policy']='baseline'

def experiment(name,sources=None,gconf=.24,tconf=.42,area=200,seam=True,
               bonus=.08,nms=.50,minfree=.45,tile_policy='baseline',fusion='none'):
    r=recipe(name,sources if sources is not None else [SRC_HI,SRC_TILES],
             gconf=gconf,tconf=tconf,area=area,seam=seam,minfree=minfree)
    r.update(engine='v23',tile_bonus=bonus,nms_iou=nms,tile_policy=tile_policy,fusion=fusion)
    return r


def modify_rle(rr,operation):
    if operation=='none':return rr
    box=coco.toBbox(rr);x,y,w,h=[int(q) for q in box]
    if w<1 or h<1:return rr
    pad=3;x0=max(0,x-pad);y0=max(0,y-pad);x1=min(W,x+w+pad);y1=min(H,y+h+pad)
    full=coco.decode(rr).astype(np.uint8)
    roi=full[y0:y1,x0:x1]
    if operation=='close3':
        result=cv2.morphologyEx(roi,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
    elif operation=='dilate1':
        result=cv2.dilate(roi,np.ones((3,3),np.uint8),iterations=1)
    elif operation=='erode1':
        result=cv2.erode(roi,np.ones((3,3),np.uint8),iterations=1)
    else:raise ValueError(operation)
    if int(result.sum())<50:return rr
    full[y0:y1,x0:x1]=result
    return enc(full)

def select_masks(raw,cfg):
    sources={tuple(pair) for pair in cfg['sources']}
    candidates=[]
    for source,items in raw.items():
        if source not in sources:continue
        threshold=cfg['tile_conf'] if source[1].startswith('t') else cfg['global_conf']
        for p in items:
            score=float(p['score']);area=int(p['area'])
            if score<threshold or area<cfg['min_area']:continue
            if cfg['seam_guard'] and source[1].startswith('t') and p.get('truncated',False):continue
            rank=score+(cfg['tile_bonus'] if source[1].startswith('t') else 0.)
            candidates.append((rank,area,p))
    candidates.sort(key=lambda q:(-q[0],-q[1]))
    masks=[];originals=[];occupied=np.zeros((H,W),np.uint8)
    for rank,area,p in candidates:
        rr=as_rle(p['counts'])
        if originals:
            ov=np.asarray(coco.iou([rr],originals,[0]*len(originals)))[0]
            if np.any(ov>=cfg['nms_iou']):continue
        mask=coco.decode(rr).astype(np.uint8)
        free=mask&(occupied==0)
        free_area=int(free.sum())
        if free_area<cfg['min_area'] or free_area/max(1,area)<cfg['min_free']:continue
        cleaned=enc(free)
        if cfg.get('refine','none')!='none':cleaned=modify_rle(cleaned,cfg['refine'])
        # Retain exact V20 occupied behavior for baseline; for refined recipes use actual refined footprint.
        if cfg.get('refine','none')!='none':
            shaped=coco.decode(cleaned).astype(np.uint8)&(occupied==0)
            if int(shaped.sum())<cfg['min_area']:continue
            cleaned=enc(shaped)
            occupied|=shaped
        else:occupied|=free
        masks.append(cleaned);originals.append(rr)
        if len(masks)>=120:break
    return masks

# Do not change the winner branch; these tests protect the baseline.
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['global_conf']==.24 and WINNER['tile_conf']==.42
assert WINNER['seam_guard'] and WINNER['min_area']==200
assert WINNER['nms_iou']==.5 and WINNER['tile_bonus']==.08
assert callable(_gc_safe.collect)

## 4. Local expert-version PQ proxy (same definition as V25)

In [ ]:
def annotation_rle(a):
    seg=a.get('segmentation')
    if seg is None or seg==[]:return None
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):
            obj=coco.frPyObjects(seg,H,W)
            return coco.merge(obj) if isinstance(obj,list) else obj
        rr=dict(seg)
        if isinstance(rr.get('counts'),str):rr['counts']=rr['counts'].encode('ascii')
        return rr
    if len(seg)>0 and isinstance(seg[0],(int,float)):seg=[seg]
    polygons=[p for p in seg if len(p)>=6 and len(p)%2==0]
    if not polygons:return None
    rr=coco.frPyObjects(polygons,H,W)
    return coco.merge(rr) if isinstance(rr,list) else rr
TRUTH={}
def gt(stem):
    if stem not in TRUTH:
        TRUTH[stem]=[[rr for a in arr if (rr:=annotation_rle(a)) is not None]
                     for _,arr in VERSIONS[stem]]
    return TRUTH[stem]
def match_counts(truth,preds):
    if not truth:return np.array([0.,0.,float(len(preds)),0.])
    if not preds:return np.array([0.,0.,0.,float(len(truth))])
    ious=np.asarray(coco.iou(truth,preds,[0]*len(preds)),float)
    # Match the earlier V20 PQ proxy EXACTLY for control-score parity.
    ii,jj=linear_sum_assignment(-ious)
    ok=ious[ii,jj]>.5
    tp=int(ok.sum())
    return np.array([float(ious[ii[ok],jj[ok]].sum()),float(tp),
                     float(len(preds)-tp),float(len(truth)-tp)])
def physical_counts(stem,preds):
    result=np.zeros(4,float)
    for version in gt(stem):result+=match_counts(version,preds)
    return result
def pq(count):
    den=count[1]+.5*(count[2]+count[3])
    return float(count[0]/den) if den>0 else 0.
def metrics(count):
    den=count[1]+.5*(count[2]+count[3])
    return {'PQ':pq(count),'SQ':float(count[0]/count[1]) if count[1] else 0.,
       'RQ':float(count[1]/den) if den else 0.,'TP':int(count[1]),
       'FP':int(count[2]),'FN':int(count[3])}
example=np.zeros((H,W),np.uint8);example[40:61,50:92]=1
assert coco.area(as_rle(txt(enc(example))))==example.sum()
assert abs(pq(match_counts([enc(example)],[enc(example)]))-1)<1e-9
print('PQ scorer and full-size RLE roundtrip: PASS')

## 5. Recover trained V24 and V25 model components; preserve the real 0.39 pipeline

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import joblib, math, time, io
from torchvision.models import resnet18

class DoubleConv(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.block=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
            nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x):return self.block(x)

class PromptUNet(nn.Module):
    def __init__(self):
        super().__init__();e=resnet18(weights=None);old=e.conv1
        e.conv1=nn.Conv2d(4,64,7,2,3,bias=False)
        self.stem=nn.Sequential(e.conv1,e.bn1,e.relu);self.pool=e.maxpool
        self.e1=e.layer1;self.e2=e.layer2;self.e3=e.layer3;self.e4=e.layer4
        self.d4=DoubleConv(768,192);self.d3=DoubleConv(320,128)
        self.d2=DoubleConv(192,96);self.d1=DoubleConv(160,64)
        self.out=nn.Sequential(nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                               nn.Conv2d(64,32,3,padding=1),nn.ReLU(inplace=True),nn.Conv2d(32,1,1))
    def forward(self,x):
        e0=self.stem(x);e1=self.e1(self.pool(e0));e2=self.e2(e1);e3=self.e3(e2);e4=self.e4(e3)
        def up(a,b):return torch.cat([F.interpolate(a,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1)
        a=self.d4(up(e4,e3));a=self.d3(up(a,e2));a=self.d2(up(a,e1));a=self.d1(up(a,e0))
        return self.out(a)

def make_channels(gray_uint8,mask_uint8):
    gray=np.asarray(gray_uint8,dtype=np.uint8)
    equal=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(gray)
    blur=cv2.GaussianBlur(gray,(3,3),0)
    dx=cv2.Sobel(blur,cv2.CV_32F,1,0,ksize=3);dy=cv2.Sobel(blur,cv2.CV_32F,0,1,ksize=3)
    edge=np.clip(cv2.magnitude(dx,dy)/240.,0,1)
    return np.stack([gray/255.,equal/255.,mask_uint8.astype(np.float32),edge],axis=0).astype(np.float32)

def normalized_bbox_roi(box,scale=1.48,min_extent=120):
    x,y,w,h=map(float,box)
    if w<=0 or h<=0:return None
    dx=min(W,max(float(min_extent),w*scale));dy=min(H,max(float(min_extent),h*scale))
    cx=x+w/2;cy=y+h/2
    x0=int(np.clip(round(cx-dx/2),0,W-1));y0=int(np.clip(round(cy-dy/2),0,H-1))
    x1=int(np.clip(round(cx+dx/2),x0+1,W));y1=int(np.clip(round(cy+dy/2),y0+1,H))
    return x0,y0,x1,y1

def find_original_asset(filename):
    direct=sorted(INPUT.rglob(filename))
    for p in direct:
        if p.is_file() and p.stat().st_size>100:return p.read_bytes(),str(p)
    for z in sorted(INPUT.rglob('*.zip')):
        if not any(token in z.name.lower() for token in ('v24','v25','v26','model','trained','prediction','report')):continue
        try:
            with zipfile.ZipFile(z) as arc:
                matches=[m for m in arc.namelist() if Path(m).name==filename]
                if matches:return arc.read(matches[0]),str(z)+'::'+matches[0]
                # V24 full output bundles a second ZIP containing its scored CSVs.
                for nested in [m for m in arc.namelist() if m.lower().endswith('.zip')]:
                    try:
                        with zipfile.ZipFile(io.BytesIO(arc.read(nested))) as inner:
                            hits=[x for x in inner.namelist() if Path(x).name==filename]
                            if hits:return inner.read(hits[0]),str(z)+'::'+nested+'::'+hits[0]
                    except (OSError,zipfile.BadZipFile):pass
        except (OSError,zipfile.BadZipFile):continue
    return None,None

for filename in ('SOLAR_V24_REFINER_BEST.pt','SOLAR_V24_INSTANCE_QUALITY.joblib'):
    target=OUT/'models'/filename
    data,source=find_original_asset(filename)
    if data is None:
        if REQUIRE_V24_MODELS:raise FileNotFoundError('Attach V24 model archive; missing '+filename)
    else:
        if not target.exists() or sha(target)!=hashlib.sha256(data).hexdigest():target.write_bytes(data)
        print('Recovered V24 asset:',filename, 'from',source,'SHA:',sha(target)[:16])

V24_REFINER_PATH=OUT/'models'/'SOLAR_V24_REFINER_BEST.pt'
V24_QUALITY_PATH=OUT/'models'/'SOLAR_V24_INSTANCE_QUALITY.joblib'
refiner_v24=PromptUNet().to('cuda:'+str(GPU_ID))
refiner_v24.load_state_dict(torch.load(V24_REFINER_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_v24.eval()
meta_v24=joblib.load(V24_QUALITY_PATH)
quality_v24=meta_v24['model'];V24_FEATURES=meta_v24['features']
assert len(V24_FEATURES)==15 and callable(quality_v24.predict_proba)
print('Recovered genuine V24 quality model, OOF AUROC:',meta_v24.get('oof_auc'))

In [ ]:
def matching_scores(rr,records):
    if not records:return 0.,0.,0.
    rles=[as_rle(q['counts']) for q in records]
    ious=np.asarray(coco.iou([rr],rles,[0]*len(rles)),float).reshape(-1)
    i=int(ious.argmax());return float(ious[i]),float(records[i]['score']),float(max(q['score'] for q in records))

@torch.inference_mode()
def run_refiner(model,gray,rr,threshold=.50,min_prompt_iou=.54):
    crop=normalized_bbox_roi([int(v) for v in coco.toBbox(rr)],scale=1.48)
    if crop is None:return rr
    x0,y0,x1,y1=crop;original=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    if not original.any():return rr
    image=cv2.resize(gray[y0:y1,x0:x1],(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_AREA)
    prompt=cv2.resize(original,(CROP_SIDE,CROP_SIDE),interpolation=cv2.INTER_NEAREST)
    inp=torch.from_numpy(make_channels(image,prompt))[None].cuda(GPU_ID)
    with torch.autocast('cuda',dtype=torch.float16):
        probabilities=model(inp).sigmoid()[0,0].float().cpu().numpy()
    refined=(cv2.resize(probabilities,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)>=threshold).astype(np.uint8)
    oa=int(original.sum());na=int(refined.sum())
    if na<80 or not (.52<=na/max(oa,1)<=1.65):return rr
    overlap=int((refined&original).sum())/max(1,oa+na-int((refined&original).sum()))
    if overlap<min_prompt_iou:return rr
    full=np.zeros((H,W),np.uint8);full[y0:y1,x0:x1]=refined
    return enc(full)

def load_base(stem):
    raw={SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    return raw,select_masks(raw,WINNER)

def v24_refined_masks(stem,baseline,gray=None):
    key=sha(V24_REFINER_PATH)[:18];p=OUT/'refined'/('v24_'+key+'_'+stem+'.json.gz')
    if p.is_file():
        with gzip.open(p,'rt') as f:v=json.load(f)
        if len(v)==len(baseline):return [as_rle(x) for x in v]
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    proposed=[run_refiner(refiner_v24,gray,rr) for rr in baseline]
    occupied=np.zeros((H,W),np.uint8);final=[]
    for a,b in zip(baseline,proposed):
        f=coco.decode(b).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:f=coco.decode(a).astype(np.uint8)&(occupied==0)
        if int(f.sum())<80:continue
        final.append(enc(f));occupied|=f
    if len(final)!=len(baseline):final=baseline
    p.parent.mkdir(parents=True,exist_ok=True)
    with gzip.open(p,'wt',compresslevel=3) as f:json.dump([txt(v) for v in final],f)
    return final

def features_v24(stem,base,refined,gray=None,raw=None):
    raw=raw if raw is not None else {SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    rows=[]
    for rr,new in zip(base,refined):
        x,y,w,h=map(float,coco.toBbox(rr));mask=coco.decode(rr).astype(np.uint8);area=int(mask.sum())
        dilation=cv2.dilate(mask,np.ones((11,11),np.uint8))
        ring=(dilation>0)&(mask==0)
        inside=float(gray[mask>0].mean()) if area else 0.;outside=float(gray[ring].mean()) if ring.any() else inside
        ga,gscore,gmax=matching_scores(rr,raw[SRC_HI]);ta,tscore,tmax=matching_scores(rr,raw[SRC_TILES])
        na=float(coco.area(new));oa=float(coco.area(rr))
        riou=float(coco.iou([rr],[new],[0])[0,0])
        rows.append({'base_score':max(gscore,tscore),'base_area_log':float(np.log1p(area)),
                     'base_box_ar':float(np.log((w+1)/(h+1))),'base_fill':float(area/max(1,w*h)),
                     'base_center_x':float((x+w/2)/W),'base_center_y':float((y+h/2)/H),
                     'global_agreement':ga,'tile_agreement':ta,'global_maxconf':gmax,'tile_maxconf':tmax,
                     'pixel_avg_inside':inside/255.,'pixel_avg_ring':outside/255.,
                     'contrast':(outside-inside)/255.,'refined_iou':riou,'refined_area_ratio':na/max(oa,1.)})
    return pd.DataFrame(rows,columns=V24_FEATURES)

def v24_control(stem):
    raw,base=load_base(stem)
    if not base:return []
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=v24_refined_masks(stem,base,gray)
    df=features_v24(stem,base,refined,gray,raw)
    qs=quality_v24.predict_proba(df[V24_FEATURES].fillna(0.))[:,1]
    return [rr for rr,q in zip(base,qs) if q>=.25]  # Exact V24 QUALITY_025: no extra clipping

s=np.zeros((H,W),np.uint8);s[50:90,70:130]=1
assert int(coco.area(enc(s)))==2400
assert WINNER['sources']==[['portable_global_m','g1792'],['portable_native_m','t1280']]
assert WINNER['tile_conf']==.42 and WINNER['global_conf']==.24 and WINNER['seam_guard']
print('V24 control frozen. New V25 scoring cannot alter its thresholds or masks.')

In [ ]:
# Restore all genuinely trained V25 checkpoint bytes; no substitute/pretrained-only networks.
from skimage.morphology import skeletonize
from skimage.segmentation import watershed

V25_REAL_PATH=OUT/'models'/'SOLAR_V25_REAL_REFINER_BEST.pt'
V25_CURVE_PATH=OUT/'models'/'SOLAR_V25_CURVE_BEST.pt'
V25_QUALITY_PATH=OUT/'models'/'SOLAR_V25_EXTENDED_QUALITY.joblib'
for p in (V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH):
    data,source=find_original_asset(p.name)
    if data is None:raise FileNotFoundError('Missing '+p.name+'. Attach the complete V25 trained checkpoint ZIP.')
    if not p.exists() or sha(p)!=hashlib.sha256(data).hexdigest():p.write_bytes(data)
    print('Recovered frozen V25',p.name,sha(p)[:18], 'from',source)
refiner_real=PromptUNet().cuda(GPU_ID)
refiner_real.load_state_dict(torch.load(V25_REAL_PATH,map_location='cpu',weights_only=False)['model'],strict=True)
refiner_real.eval();REAL_REFINER_SHA=sha(V25_REAL_PATH)

def real_refine(stem,rr,gray=None):
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    return run_refiner(refiner_real,gray,rr,threshold=.5,min_prompt_iou=.50)

v25_ranker_package=joblib.load(V25_QUALITY_PATH)
new_rankers=v25_ranker_package['models']
V25_QUALITY_CUTOFF=float(v25_ranker_package['cutoff'])
print('Frozen V25 ranker cutoff:',V25_QUALITY_CUTOFF,'feature count:',len(v25_ranker_package['columns']))
assert abs(V25_QUALITY_CUTOFF-V25_EXPECTED_QUALITY_CUTOFF)<1e-6, 'Wrong V25 ranking checkpoint/cutoff'

def rank_probs(models,df):
    if df.empty:return np.asarray([],float)
    a=df[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)
    return np.mean([m.predict_proba(a)[:,1] for m in models],axis=0)


In [ ]:
class MiniUNet(nn.Module):
    def __init__(self,channels=2,outputs=3):
        super().__init__()
        self.a=DoubleConv(channels,24);self.b=DoubleConv(24,48);self.c=DoubleConv(48,96)
        self.d=DoubleConv(96,160)
        self.up3=DoubleConv(160+96,96);self.up2=DoubleConv(96+48,64)
        self.up1=DoubleConv(64+24,32);self.final=nn.Conv2d(32,outputs,1)
    def forward(self,x):
        a=self.a(x);b=self.b(F.avg_pool2d(a,2));c=self.c(F.avg_pool2d(b,2));d=self.d(F.avg_pool2d(c,2))
        u=self.up3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        u=self.up2(torch.cat([F.interpolate(u,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        u=self.up1(torch.cat([F.interpolate(u,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return self.final(u)


curve_path=V25_CURVE_PATH
curve_net=MiniUNet().cuda(GPU_ID)
curve_net.load_state_dict(torch.load(curve_path,map_location='cpu',weights_only=False)['model'],strict=True)
curve_net.eval();CURVE_SHA=sha(curve_path)
print('Loaded frozen V25 curve network',CURVE_SHA[:18])

In [ ]:
def curve_locations():
    starts=list(range(0,H-CURVE_WINDOW+1,CURVE_STRIDE))
    if starts[-1]!=H-CURVE_WINDOW:starts.append(H-CURVE_WINDOW)
    return [(x,y) for y in starts for x in starts]
CURVE_LOCATIONS=curve_locations()
assert len(CURVE_LOCATIONS)==9, CURVE_LOCATIONS

def curve_input(gray):
    equ=cv2.createCLAHE(clipLimit=2.,tileGridSize=(8,8)).apply(gray)
    return torch.from_numpy(np.stack([gray/255.,equ/255.],0).astype(np.float32))[None].cuda(GPU_ID)

@torch.inference_mode()
def curve_proposals(stem):
    cache=OUT/'curve_raw'/(stem+'.json.gz')
    signature=hashlib.sha256((CURVE_SHA+stem+str(CURVE_WINDOW)+str(CURVE_SIDE)).encode()).hexdigest()
    if cache.exists():
        try:
            with gzip.open(cache,'rt') as f:v=json.load(f)
            if v.get('identity')==signature:return v['proposals']
        except (ValueError,OSError,KeyError):pass
    image=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    if image is None:raise FileNotFoundError(stem)
    props=[]
    for x0,y0 in CURVE_LOCATIONS:
        raw=image[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]
        inp=cv2.resize(raw,(CURVE_SIDE,CURVE_SIDE),interpolation=cv2.INTER_AREA)
        with torch.autocast('cuda',dtype=torch.float16):
            probs=curve_net(curve_input(inp)).sigmoid()[0].float().cpu().numpy()
        fg=cv2.resize(probs[0],(CURVE_WINDOW,CURVE_WINDOW))
        center=cv2.resize(probs[1],(CURVE_WINDOW,CURVE_WINDOW))
        seg=(fg>.53).astype(np.uint8)
        n,components,stats,_=cv2.connectedComponentsWithStats(seg,connectivity=8)
        for k in range(1,n):
            component_area=int(stats[k,cv2.CC_STAT_AREA])
            if component_area<CURVE_MIN_AREA:continue
            local=(components==k).astype(np.uint8)
            # Conservatively split only when two *distinct* centerline islands exist.
            line=(center>.56).astype(np.uint8)&local
            m,markers,mark_stats,_=cv2.connectedComponentsWithStats(line,connectivity=8)
            good=[q for q in range(1,m) if mark_stats[q,cv2.CC_STAT_AREA]>=8]
            pieces=[local]
            if 2<=len(good)<=5 and component_area>700:
                seeds=np.zeros_like(local,np.int32)
                for label_idx,mark_idx in enumerate(good,1):seeds[markers==mark_idx]=label_idx
                regions=watershed(-cv2.distanceTransform(local,cv2.DIST_L2,3),seeds,mask=(local>0))
                candidates=[(regions==z).astype(np.uint8) for z in range(1,len(good)+1)]
                if all(int(c.sum())>=CURVE_MIN_AREA for c in candidates):pieces=candidates
            for part in pieces:
                area=int(part.sum())
                if area<CURVE_MIN_AREA:continue
                sy,sx=np.where(part)
                if min(sx)<8 and x0>0 or max(sx)>CURVE_WINDOW-9 and x0+CURVE_WINDOW<W:continue
                if min(sy)<8 and y0>0 or max(sy)>CURVE_WINDOW-9 and y0+CURVE_WINDOW<H:continue
                local_gray=raw[part>0];mask_score=float(np.mean(fg[part>0]))
                line_score=float(np.mean(center[part>0]))
                if mask_score<.59 or line_score<.18:continue
                # Relatively dark features, not arbitrary bright solar active-region structures.
                patch=raw[max(0,sy.min()-16):min(CURVE_WINDOW,sy.max()+17),
                          max(0,sx.min()-16):min(CURVE_WINDOW,sx.max()+17)]
                if len(local_gray) and float(np.median(local_gray))>float(np.percentile(patch,75))+8:continue
                full=np.zeros((H,W),np.uint8)
                full[y0:y0+CURVE_WINDOW,x0:x0+CURVE_WINDOW]=part
                rr=enc(full)
                props.append({'counts':txt(rr),'score':float(np.clip(.65*mask_score+.35*line_score,0,1)),
                              'area':int(coco.area(rr)),'truncated':False,'source_view':'curve_v25',
                              'center_support':line_score})
    props.sort(key=lambda p:p['score'],reverse=True)
    distinct=[];rles=[]
    for p in props:
        rr=as_rle(p['counts'])
        if rles and float(np.max(coco.iou([rr],rles,[0]*len(rles))))>.33:continue
        distinct.append(p);rles.append(rr)
        if len(distinct)>=CURVE_MAX_PROPOSALS:break
    with gzip.open(cache,'wt',compresslevel=3) as f:
        json.dump({'identity':signature,'stem':stem,'proposals':distinct},f,separators=(',',':'))
    return distinct

def extend_with_curve(stem,baseline,raw=None):
    # Existing instance identities are never silently split or replaced here.
    raw=raw if raw is not None else load_base(stem)[0]
    base_rles=list(baseline)
    base_all=base_rles.copy()
    occupied=np.zeros((H,W),np.uint8)
    for rr in base_rles:occupied|=coco.decode(rr).astype(np.uint8)
    additions=[]
    for p in curve_proposals(stem):
        rr=as_rle(p['counts'])
        if base_all and float(np.max(coco.iou([rr],base_all,[0]*len(base_all))))>=.25:continue
        part=coco.decode(rr).astype(np.uint8)
        free=part&(occupied==0)
        if int(free.sum())<CURVE_MIN_AREA or free.sum()/max(1,part.sum())<.70:continue
        accepted=enc(free)
        additions.append(accepted);occupied|=free;base_all.append(accepted)
    return base_rles+additions,['detector']*len(base_rles)+['curve']*len(additions)

print('Curve candidate generation initialized:',len(CURVE_LOCATIONS),'windows per 2048 scene')

In [ ]:
from scipy.spatial.distance import cdist

# These 40+ new features are computed from *predicted masks/images only*.
EXTRA_FIELDS=['log_perimeter','log_convex_area','solidity','extent','elongation','eccentricity',
 'aspect_absolute','area_frac_box','edge_density','endpoint_count','branchpoint_count','skeleton_length',
 'skeleton_area_ratio','mean_thickness','max_thickness','width_variation','local_component_count',
 'intensity_inside_std','intensity_inside_p10','intensity_inside_p50','intensity_inside_p90',
 'intensity_ring_p50','intensity_diff_p10','intensity_diff_mean','inside_entropy','sobel_inside_mean',
 'sobel_ring_mean','contrast_normalized','sun_radial','rim_distance','mask_near_solar_limb',
 'hull_vs_mask','compactness','hu_1','hu_2','pca_angle_sin','pca_angle_cos',
 'neighbor_center_dist','neighbor_bbox_iou','neighbor_area_ratio','n_neighbors_100',
 'curve_origin','curve_conf','curve_center_support','curve_vs_global_iou','curve_vs_tile_iou',
 'raw_global_count','raw_tile_count','raw_high_conf_global_count','raw_high_conf_tile_count',
 'mask_global_rank','mask_tile_rank','mask_area_over_disk','intensity_outside_std']
NEW_FEATURES=list(V24_FEATURES)+EXTRA_FIELDS
assert len(NEW_FEATURES)>60

def image_entropy(pixels):
    if len(pixels)==0:return 0.
    hist=np.bincount((pixels//16).astype(np.uint8),minlength=16).astype(float)
    hist=hist/max(hist.sum(),1);h=hist[hist>0]
    return float(-(h*np.log(h)).sum())

def extract_extra(gray,rr,all_boxes,source_kind,curve_meta,raw):
    x,y,w,h=map(float,coco.toBbox(rr));x0=max(0,int(x)-12);y0=max(0,int(y)-12)
    x1=min(W,int(np.ceil(x+w))+12);y1=min(H,int(np.ceil(y+h))+12)
    # COCO decode is full-frame but geometric operations are limited to a small crop.
    mask=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
    g=gray[y0:y1,x0:x1];area=int(mask.sum());yy,xx=np.where(mask>0)
    ring=cv2.dilate(mask,np.ones((9,9),np.uint8))&(1-mask)
    inside=g[mask>0];outside=g[ring>0]
    if not len(inside):inside=np.asarray([0],np.uint8)
    if not len(outside):outside=inside
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    perimeter=float(sum(cv2.arcLength(c,True) for c in contours))
    hull_area=float(sum(cv2.contourArea(cv2.convexHull(c)) for c in contours if len(c)>=3))
    # Downsample skeleton for speed; normalize pixel-to-image ratio.
    ratio=min(1.,150/max(mask.shape));sw=max(8,int(mask.shape[1]*ratio));sh=max(8,int(mask.shape[0]*ratio))
    reduced=(cv2.resize(mask,(sw,sh),interpolation=cv2.INTER_NEAREST)>0).astype(np.uint8)
    sk=skeletonize(reduced>0).astype(np.uint8)
    neigh=cv2.filter2D(sk,cv2.CV_16S,np.ones((3,3),np.uint8),borderType=cv2.BORDER_CONSTANT)-sk.astype(np.int16)
    endpoints=int(((sk>0)&(neigh==1)).sum());branches=int(((sk>0)&(neigh>=3)).sum())
    dist=cv2.distanceTransform(reduced,cv2.DIST_L2,3)[reduced>0]
    components=cv2.connectedComponents(reduced,connectivity=8)[0]-1
    if len(xx)>1:
        cov=np.cov(np.stack([xx,yy]).astype(float));ev,vec=np.linalg.eigh(cov)
        major=vec[:,np.argmax(ev)];elong=float(np.sqrt((max(ev)+1)/(min(ev)+1)))
        ecc=float(np.sqrt(max(0.,1-min(ev)/(max(ev)+1e-6))))
    else:major=np.array([1.,0.]);elong=1.;ecc=0.
    sob=cv2.magnitude(cv2.Sobel(g,cv2.CV_32F,1,0),cv2.Sobel(g,cv2.CV_32F,0,1))
    hu=cv2.HuMoments(cv2.moments(mask)).flatten()
    cx=x+w/2;cy=y+h/2;rad=np.hypot(cx-W/2,cy-H/2)/(W/2)
    other_boxes=np.asarray(all_boxes,float).reshape(-1,4)
    current_centroid=np.array([cx,cy]);centroids=other_boxes[:,:2]+other_boxes[:,2:]/2 if len(other_boxes) else np.zeros((0,2))
    distances=np.linalg.norm(centroids-current_centroid,axis=1) if len(centroids) else np.asarray([])
    nearby=sorted(float(z) for z in distances if z>1)
    bbox_ious=[]
    for bx,by,bw,bh in other_boxes:
        if abs(bx-x)+abs(by-y)+abs(bw-w)+abs(bh-h)<.001:continue
        iw=max(0,min(x+w,bx+bw)-max(x,bx));ih=max(0,min(y+h,by+bh)-max(y,by))
        bbox_ious.append(iw*ih/max(1,w*h+bw*bh-iw*ih))
    raw_g=raw[SRC_HI];raw_t=raw[SRC_TILES]
    gi,gc,gmax=matching_scores(rr,raw_g);ti,tc,tmax=matching_scores(rr,raw_t)
    flat=dict(log_perimeter=np.log1p(perimeter),log_convex_area=np.log1p(hull_area),
        solidity=area/max(1,hull_area),extent=area/max(1,w*h),elongation=min(100,elong),
        eccentricity=ecc,aspect_absolute=max((w+1)/(h+1),(h+1)/(w+1)),area_frac_box=area/max(1,w*h),
        edge_density=perimeter/max(1,area),endpoint_count=endpoints,branchpoint_count=branches,
        skeleton_length=int(sk.sum())/max(.15,ratio),skeleton_area_ratio=float(sk.sum())/max(1,reduced.sum()),
        mean_thickness=float(np.mean(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        max_thickness=float(np.max(dist)*2/max(.1,ratio)) if len(dist) else 0.,
        width_variation=float(np.std(dist)) if len(dist) else 0.,local_component_count=max(0,components),
        intensity_inside_std=float(inside.std())/255.,intensity_inside_p10=float(np.percentile(inside,10))/255.,
        intensity_inside_p50=float(np.median(inside))/255.,intensity_inside_p90=float(np.percentile(inside,90))/255.,
        intensity_ring_p50=float(np.median(outside))/255.,
        intensity_diff_p10=float(np.percentile(outside,10)-np.percentile(inside,10))/255.,
        intensity_diff_mean=float(outside.mean()-inside.mean())/255.,inside_entropy=image_entropy(inside),
        sobel_inside_mean=float(sob[mask>0].mean())/255. if area else 0.,
        sobel_ring_mean=float(sob[ring>0].mean())/255. if ring.any() else 0.,
        contrast_normalized=float(outside.mean()-inside.mean())/max(10.,float(outside.std())),
        sun_radial=rad,rim_distance=max(0,1-rad),mask_near_solar_limb=float(rad>.93),
        hull_vs_mask=hull_area/max(1,area),compactness=perimeter**2/max(1,4*np.pi*area),
        hu_1=float(np.sign(hu[0])*np.log1p(abs(hu[0]))),hu_2=float(np.sign(hu[1])*np.log1p(abs(hu[1]))),
        pca_angle_sin=float(major[1]),pca_angle_cos=float(major[0]),
        neighbor_center_dist=nearby[0]/2048 if nearby else 1.,
        neighbor_bbox_iou=max(bbox_ious,default=0.),
        neighbor_area_ratio=float(sorted([abs(np.log((w*h+1)/(bw*bh+1))) for bx,by,bw,bh in other_boxes])[1]) if len(other_boxes)>1 else 0.,
        n_neighbors_100=sum(v<100 for v in nearby),curve_origin=float(source_kind=='curve'),
        curve_conf=float(curve_meta.get('score',0)),curve_center_support=float(curve_meta.get('center_support',0)),
        curve_vs_global_iou=gi,curve_vs_tile_iou=ti,
        raw_global_count=len(raw_g),raw_tile_count=len(raw_t),
        raw_high_conf_global_count=sum(p['score']>.4 for p in raw_g),
        raw_high_conf_tile_count=sum(p['score']>.4 for p in raw_t),
        mask_global_rank=gc,mask_tile_rank=tc,mask_area_over_disk=area/(np.pi*1024**2),
        intensity_outside_std=float(outside.std())/255.)
    assert set(flat)==set(EXTRA_FIELDS),('Missing feature',set(EXTRA_FIELDS)-set(flat))
    return flat

def extended_features(stem,masks,origins=None,curve_meta=None,gray=None,raw=None):
    if not masks:return pd.DataFrame(columns=NEW_FEATURES)
    if gray is None:gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    if raw is None:raw=load_base(stem)[0]
    # Refinement-based V24 features need the frozen, correct V24 refiner.
    if origins is None:origins=['detector']*len(masks)
    if curve_meta is None:curve_meta=[{} for _ in masks]
    assert len(masks)==len(origins)==len(curve_meta)
    refined=v24_refined_masks(stem,masks,gray) if all(o=='detector' for o in origins) else [
        run_refiner(refiner_v24,gray,r) if o=='detector' else r for r,o in zip(masks,origins)]
    base=features_v24(stem,masks,refined,gray,raw)
    boxes=[list(map(float,coco.toBbox(rr))) for rr in masks]
    for i,rr in enumerate(masks):
        extra=extract_extra(gray,rr,boxes,origins[i],curve_meta[i],raw)
        for k in EXTRA_FIELDS:base.loc[i,k]=extra[k]
    return base[NEW_FEATURES].replace([np.inf,-np.inf],0).fillna(0.)

print('Extended feature count:',len(NEW_FEATURES))

In [ ]:
VAL_STEMS=sorted(VALIDATION)
assert len(VAL_STEMS)==104 or not FULL_VALIDATION_REQUIRED
by_day=defaultdict(list)
for st in VAL_STEMS:by_day[st[:8]].append(st)
all_days=sorted(by_day);random.Random(SEED+952).shuffle(all_days)
CAL_DAYS=set(all_days[:int(len(all_days)*.52)])
CAL=sorted(s for s in VAL_STEMS if s[:8] in CAL_DAYS)
AUDIT=sorted(s for s in VAL_STEMS if s[:8] not in CAL_DAYS)
assert len(CAL)>30 and len(AUDIT)>30 and not ({s[:8] for s in CAL}&{s[:8] for s in AUDIT})
print('Train quality on',len(CAL),'scenes, evaluate on',len(AUDIT),'different-day scenes')

# Cache masks and features separately from the checkpointed models.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:
        full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    meta=[]
    for index,rr in enumerate(full):
        if origins[index]=='detector':meta.append({})
        else:
            index_ious=np.asarray(coco.iou([rr],[as_rle(v['counts']) for v in curve_list],
                                             [0]*len(curve_list)),float).reshape(-1) if curve_list else np.asarray([])
            meta.append(curve_list[int(index_ious.argmax())] if len(index_ious) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    path=OUT/'reports'/('features_'+stem+'.joblib')
    # Data are caches of MODEL PREDICTIONS, not ground-truth test labels.
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if path.exists():
        try:
            obj=joblib.load(path)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    obj={'signature':signature,'masks':[txt(x) for x in masks], 'origins':origins,
         'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(obj,path,compress=3)
    return obj

def consensus_quality_labels(stem,preds):
    expert_versions=gt(stem);out=[]
    for rr in preds:
        votes=[]
        for truth in expert_versions:
            if truth:
                vec=np.asarray(coco.iou([rr],truth,[0]*len(truth)),float).reshape(-1)
                votes.append(float(np.max(vec)) if len(vec) else 0.)
            else:votes.append(0.)
        out.append(int(np.mean(np.asarray(votes)>.5)>=.5) if votes else 0)
    return out


assert list(v25_ranker_package['columns'])==NEW_FEATURES,'V25 feature order mismatch'


In [ ]:
# Use all frozen V25 components to reconstruct its exact candidate-order/quality behavior.
def scene_candidates(stem,include_curve=True):
    raw,base=load_base(stem)
    if include_curve:full,origins=extend_with_curve(stem,base,raw)
    else:full,origins=base,['detector']*len(base)
    curve_list=curve_proposals(stem) if include_curve else []
    curve_rles=[as_rle(p['counts']) for p in curve_list]
    meta=[]
    for i,rr in enumerate(full):
        if origins[i]=='detector':meta.append({});continue
        overlaps=np.asarray(coco.iou([rr],curve_rles,[0]*len(curve_rles)),float).reshape(-1) if curve_rles else np.asarray([])
        meta.append(curve_list[int(overlaps.argmax())] if len(overlaps) else {})
    return raw,full,origins,meta,len(base)

def scene_feature_data(stem):
    location=OUT/'reports'/('v25_features_'+stem+'.joblib')
    signature=hashlib.sha256((CURVE_SHA+REAL_REFINER_SHA+sha(V24_REFINER_PATH)+sha(V25_QUALITY_PATH)+str(NEW_FEATURES)).encode()).hexdigest()
    if location.is_file():
        try:
            obj=joblib.load(location)
            if obj.get('signature')==signature:return obj
        except Exception:pass
    raw,masks,origins,meta,n_base=scene_candidates(stem)
    feats=extended_features(stem,masks,origins,meta,raw=raw)
    result={'signature':signature,'masks':[txt(r) for r in masks], 'origins':origins,
            'n_base':n_base,'features':feats,'stem':stem}
    joblib.dump(result,location,compress=3)
    return result

def nonoverlap_after_refinement(masks):
    occupied=np.zeros((H,W),np.uint8);out=[]
    for rr in masks:
        pix=coco.decode(rr).astype(np.uint8)&(occupied==0)
        if int(pix.sum())<80:continue
        out.append(enc(pix));occupied|=pix
    return out

def frozen_v25_masks(stem, obj=None):
    info=obj if obj is not None else scene_feature_data(stem)
    prob=rank_probs(new_rankers,info['features'])
    inds=[i for i in range(info['n_base']) if prob[i]>=V25_QUALITY_CUTOFF]
    masks=[as_rle(info['masks'][i]) for i in inds]
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),cv2.IMREAD_GRAYSCALE)
    refined=[real_refine(stem,rr,gray) if prob[i]>=.50 else rr for i,rr in zip(inds,masks)]
    return nonoverlap_after_refinement(refined)

assert V25_QUALITY_CUTOFF==V25_EXPECTED_QUALITY_CUTOFF
assert list(v25_ranker_package['columns'])==NEW_FEATURES
print('V25 0.39 candidate logic restored, original ranker/curve/refiner frozen.')


## 6. Recover byte-identical 0.39 submission before experimenting

This is deliberately done **before** model training or test export. The previously scored V25 CSV must match its known SHA prefix. In new CSVs the V25 anchors are decoded from this exact file, avoiding a known V25 regeneration hash discrepancy. Local validation still uses the frozen model's known reconstruction; that limitation is disclosed in the report.


In [ ]:
import csv
PROVEN=Path('/kaggle/working/V29_PROTECTED_V25_PUBLIC_039.csv')
found=False
for filename in ('V25_REAL_REFINE_PLUS_QUALITY.csv','submission.csv'):
    data,source=find_original_asset(filename)
    if data is None:continue
    if hashlib.sha256(data).hexdigest().startswith(V25_EXPECTED_CSV_SHA):
        PROVEN.write_bytes(data);found=True;print('Verified historical file from:',source);break
assert found and sha(PROVEN).startswith(V25_EXPECTED_CSV_SHA),(
    'Attach the *original* V25 score-0.39 CSV or SOLAR_V25_ALL_PREDICTIONS_AND_REPORTS.zip. '
    'The new method will not pretend that a regenerated baseline is already proven.')
proven_frame=pd.read_csv(PROVEN,dtype=str)
assert len(proven_frame)==V25_EXPECTED_ROWS
PROVEN_BY_STEM=defaultdict(list)
for row in proven_frame.itertuples(index=False):
    stem=row.filament_id.rsplit('_',1)[0]
    PROVEN_BY_STEM[stem].append(as_rle(row.segmentation_rle))
assert set(PROVEN_BY_STEM).issubset(set(TEST))
print('Exact 0.39 control protected:',sha(PROVEN)[:20],len(proven_frame),'instances')


## 7. Candidate mining from previously rejected detections + high-confidence curvilinear candidates

Instead of training another giant detector, mine **missed-object candidates** from the global and native-tile model predictions before V25's high-confidence and seam filters. Reject masks overlapping accepted V25 instances; retain instance IDs and model support. Propose a small number of two-fragment unions only if the parts are geometrically plausible. No merge is applied to V25's existing masks without evidence.


In [ ]:
from scipy.ndimage import distance_transform_edt
from scipy.spatial.distance import cdist

def overlap_max(rr,other):
    if not other:return 0.
    return float(np.max(coco.iou([rr],other,[0]*len(other))))

def mask_axis(rr):
    x,y,w,h=map(float,coco.toBbox(rr));left=int(x);top=int(y)
    full=coco.decode(rr);pix=full[top:min(H,top+int(h)+2),left:min(W,left+int(w)+2)]
    yy,xx=np.where(pix>0)
    if len(xx)<12:return np.asarray([[x,y],[x+w,y+h]],float),np.array([1.,0.],float)
    points=np.stack([xx+left,yy+top],1).astype(float)
    if len(points)>1400:points=points[::max(1,len(points)//1200)]
    cov=np.cov(points.T);vals,vecs=np.linalg.eigh(cov);axis=vecs[:,np.argmax(vals)]
    projected=points@axis
    return np.stack([points[np.argmin(projected)],points[np.argmax(projected)]]),axis

def candidate_pool(stem,anchors=None):
    anchors=list(anchors) if anchors is not None else frozen_v25_masks(stem)
    # Same anchor logic in validation and test; exact test anchors may be loaded later.
    cache=OUT/'candidate_cache'/(stem+'.joblib')
    signature=hashlib.sha256((stem+sha(V25_QUALITY_PATH)+CURVE_SHA+
          str((LOW_GLOBAL,LOW_TILE,LOW_CURVE,MAX_SINGLES_PER_SCENE,MAX_FRAGMENT_PAIRS))+
          hashlib.sha256(''.join(txt(x) for x in anchors).encode()).hexdigest()).encode()).hexdigest()
    if cache.exists():
        try:
            z=joblib.load(cache)
            if z.get('signature')==signature:return z
        except Exception:pass
    raw,base=load_base(stem)
    anchors_rle=list(anchors)
    candidates=[];seen=[]
    streams=[]
    for source in (SRC_HI,SRC_TILES):
        threshold=LOW_TILE if source==SRC_TILES else LOW_GLOBAL
        limit=MAX_TILE_RAW if source==SRC_TILES else MAX_GLOBAL_RAW
        for item in sorted(raw[source],key=lambda p:-float(p['score']))[:limit]:
            if float(item['score'])>=threshold and int(item['area'])>=MIN_EXTRA_AREA:
                streams.append((item,source[1]))
    # V25's curve model is NOT trusted to add instances by itself; only nominate candidates.
    for item in curve_proposals(stem)[:MAX_CURVE_RAW]:
        if float(item['score'])>=LOW_CURVE and int(item['area'])>=MIN_EXTRA_AREA:
            streams.append((item,'curve'))
    streams.sort(key=lambda p:-float(p[0]['score']))
    for item,source in streams:
        rr=as_rle(item['counts']);a=int(coco.area(rr))
        if a<MIN_EXTRA_AREA:continue
        if overlap_max(rr,anchors_rle)>=MAX_ACCEPTED_OVERLAP:continue
        if overlap_max(rr,seen)>=.38:continue
        # Keep raw mask shape; also encode tiled fragments with source/seam features for ranking.
        candidates.append({'counts':txt(rr),'score':float(item['score']),
            'origin':source,'kind':'single','area':a,'truncated':bool(item.get('truncated',False)),
            'center_support':float(item.get('center_support',0.)),'gap':0.})
        seen.append(rr)
        if len(candidates)>=MAX_SINGLES_PER_SCENE:break
    # Pair incomplete, nearby elongated masks, preserving both as ONE instance candidate.
    # Their union can be disjoint; unlike classical morphological closing, no fabricated
    # bridge pixels are introduced. Scoring must establish that they belong together.
    pair_options=[]
    basis=candidates[:18]
    desc=[mask_axis(as_rle(c['counts'])) for c in basis]
    for i in range(len(basis)):
        for j in range(i+1,len(basis)):
            ri=as_rle(basis[i]['counts']);rj=as_rle(basis[j]['counts'])
            if overlap_max(ri,[rj])>PAIR_MAX_IOU:continue
            p1,v1=desc[i];p2,v2=desc[j]
            dst=float(np.linalg.norm(p1[:,None,:]-p2[None,:,:],axis=-1).min())
            align=float(abs(v1@v2))
            if dst>PAIR_END_DISTANCE or align<.52:continue
            if max(int(basis[i]['area']),int(basis[j]['area']))>5*min(int(basis[i]['area']),int(basis[j]['area'])):continue
            pair_options.append((dst,1-align,i,j))
    for dst,_,i,j in sorted(pair_options)[:MAX_FRAGMENT_PAIRS]:
        ri=as_rle(basis[i]['counts']);rj=as_rle(basis[j]['counts']);rr=coco.merge([ri,rj])
        if overlap_max(rr,anchors_rle)>=MAX_ACCEPTED_OVERLAP:continue
        candidates.append({'counts':txt(rr),'score':float(min(basis[i]['score'],basis[j]['score'])),
            'origin':'paired','kind':'pair','area':int(coco.area(rr)),'truncated':False,
            'center_support':0.,'gap':float(dst)})
    result={'signature':signature,'candidates':candidates,'anchor_count':len(anchors)}
    joblib.dump(result,cache,compress=3)
    return result

# One-to-one PQ identity: denominator 0.5*(N_pred+N_ground_truth) for any matched count.
for tp,fp,fn in ((2,3,5),(9,1,0),(0,0,12)):
    assert tp+.5*(fp+fn)==.5*((tp+fp)+(tp+fn))
print('PASS: PQ denominator identity; two-source candidate and fragment pipeline ready')


## 8. Distinct novelty-credit labels and 35 image/mask/neighbor features

A candidate only receives credit if its predicted mask matches a GT instance at IoU > 0.5 **and that expert instance was not already matched by V25**. This prevents a large valid-but-duplicate mask from being mislabeled as an improvement. The amount of credit is average matched IoU across annotators. Extra features are computed from the image and model predictions only, with no metadata about hidden test labels.


In [ ]:
COLUMNS=['confidence','area_log','bbox_w_log','bbox_h_log','aspect_log','fill','solidity','compactness',
    'elongation','eccentricity','mean_width','skeleton_len','endpoint_count','branch_count',
    'inner_mean','inner_std','inner_p10','outer_mean','contrast','gradient_mean','edge_contrast',
    'max_anchor_iou','nearest_anchor_dist','global_agreement','tile_agreement','raw_global_count',
    'raw_tile_count','radial_position','is_curve','is_tile','is_pair','tile_seam','fragment_gap',
    'local_occupancy','source_agreement_diff']

def candidate_features(stem,props,anchors):
    if not props:return pd.DataFrame(columns=COLUMNS)
    gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
    assert gray is not None
    raw={SRC_HI:raw_predictions(SRC_HI,stem),SRC_TILES:raw_predictions(SRC_TILES,stem)}
    raw_g=[as_rle(item['counts']) for item in raw[SRC_HI]]
    raw_t=[as_rle(item['counts']) for item in raw[SRC_TILES]]
    aboxes=np.asarray([coco.toBbox(x) for x in anchors],float).reshape(-1,4)
    acenters=aboxes[:,:2]+aboxes[:,2:]/2 if len(aboxes) else np.zeros((0,2))
    rows=[]
    for item in props:
        rr=as_rle(item['counts']);x,y,w,h=map(float,coco.toBbox(rr))
        x0=max(0,int(x)-9);y0=max(0,int(y)-9)
        x1=min(W,int(np.ceil(x+w))+9);y1=min(H,int(np.ceil(y+h))+9)
        small=coco.decode(rr)[y0:y1,x0:x1].astype(np.uint8)
        g=gray[y0:y1,x0:x1];area=int(small.sum())
        ring=cv2.dilate(small,np.ones((7,7),np.uint8))&(small==0)
        inside=g[small>0].astype(float);outside=g[ring>0].astype(float)
        if not len(inside):inside=np.asarray([0.])
        if not len(outside):outside=inside
        contours,_=cv2.findContours(small,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        perimeter=float(sum(cv2.arcLength(c,True) for c in contours))
        hull=float(sum(cv2.contourArea(cv2.convexHull(c)) for c in contours if len(c)>=3))
        ratio=min(1.,136/max(small.shape));dst=(max(10,int(small.shape[1]*ratio)),max(10,int(small.shape[0]*ratio)))
        shr=(cv2.resize(small,dst,interpolation=cv2.INTER_NEAREST)>0).astype(np.uint8)
        sk=skeletonize(shr>0).astype(np.uint8)
        neighbors=cv2.filter2D(sk,cv2.CV_16S,np.ones((3,3),np.uint8))-sk.astype(np.int16)
        ys,xs=np.where(small)
        if len(xs)>2:
            eigen=np.linalg.eigvalsh(np.cov(np.stack([xs,ys]).astype(float)))
            elong=float(np.sqrt((eigen.max()+1)/(eigen.min()+1)))
            ecc=float(np.sqrt(max(0.,1-eigen.min()/(eigen.max()+1e-6))))
        else:elong=1.;ecc=0.
        diff=cv2.magnitude(cv2.Sobel(g,cv2.CV_32F,1,0),cv2.Sobel(g,cv2.CV_32F,0,1))
        near=np.linalg.norm(acenters-np.asarray([x+w/2,y+h/2]),axis=1).min()/2048 if len(acenters) else 1.
        overlap=overlap_max(rr,anchors)
        ga=overlap_max(rr,raw_g);ta=overlap_max(rr,raw_t)
        line_dist=cv2.distanceTransform(shr,cv2.DIST_L2,3)
        widths=line_dist[shr>0]
        rows.append({'confidence':item['score'],'area_log':np.log1p(area),
            'bbox_w_log':np.log1p(w),'bbox_h_log':np.log1p(h),
            'aspect_log':np.log((w+1)/(h+1)),'fill':area/max(1,w*h),
            'solidity':area/max(1,hull),'compactness':perimeter**2/max(1,4*np.pi*area),
            'elongation':elong,'eccentricity':ecc,'mean_width':float(2*widths.mean()/max(.1,ratio)) if len(widths) else 0.,
            'skeleton_len':float(sk.sum()/max(.1,ratio)),
            'endpoint_count':int(((sk>0)&(neighbors==1)).sum()),
            'branch_count':int(((sk>0)&(neighbors>=3)).sum()),
            'inner_mean':inside.mean()/255.,'inner_std':inside.std()/255.,
            'inner_p10':float(np.percentile(inside,10))/255.,'outer_mean':outside.mean()/255.,
            'contrast':(outside.mean()-inside.mean())/255.,
            'gradient_mean':float(diff[small>0].mean())/255. if area else 0.,
            'edge_contrast':float(diff[ring>0].mean())/255. if ring.any() else 0.,
            'max_anchor_iou':overlap,'nearest_anchor_dist':near,
            'global_agreement':ga,'tile_agreement':ta,
            'raw_global_count':len(raw_g),'raw_tile_count':len(raw_t),
            'radial_position':float(np.hypot(x+w/2-W/2,y+h/2-H/2)/1024),
            'is_curve':float(item['origin']=='curve'),
            'is_tile':float(item['origin']=='t1280'),
            'is_pair':float(item['kind']=='pair'),
            'tile_seam':float(item['truncated']), 'fragment_gap':float(item['gap'])/100.,
            'local_occupancy':float(len(anchors))/25.,
            'source_agreement_diff':abs(ga-ta)})
    frame=pd.DataFrame(rows,columns=COLUMNS).replace([np.inf,-np.inf],0.).fillna(0.)
    assert len(frame)==len(props)
    return frame

def expert_novelty_credits(stem,proposals,old):
    # Assign V25 detections to each expert using the same IoU > .5 rule as the scorer.
    reference=gt(stem)
    responses=[]
    candidate_rles=[as_rle(p['counts']) for p in proposals]
    for version in reference:
        if not version:
            responses.append(np.zeros(len(proposals),float));continue
        covered=set()
        if old:
            old_ious=np.asarray(coco.iou(version,old,[0]*len(old)),float)
            ii,jj=linear_sum_assignment(-old_ious)
            covered={int(k) for k,j in zip(ii,jj) if old_ious[k,j]>.5}
        if candidate_rles:
            mat=np.asarray(coco.iou(candidate_rles,version,[0]*len(version)),float)
            scores=[]
            for row in mat:
                best=int(row.argmax());value=float(row[best]);scores.append(value if value>.5 and best not in covered else 0.)
        else:scores=[]
        responses.append(np.asarray(scores,float))
    return np.stack(responses).mean(0) if responses else np.zeros(len(proposals),float)

def scene_data(stem,with_labels=False,anchors=None):
    anchors=list(anchors) if anchors is not None else frozen_v25_masks(stem)
    pool=candidate_pool(stem,anchors)['candidates']
    # This cache holds features only; labels can be read ONLY for train/calibration scenes.
    digest=hashlib.sha256((stem+sha(V25_QUALITY_PATH)+sha(V25_REAL_PATH)+
        str((LOW_GLOBAL,LOW_TILE,LOW_CURVE,MAX_FRAGMENT_PAIRS))+
        hashlib.sha256(''.join(txt(x) for x in anchors).encode()).hexdigest()).encode()).hexdigest()
    location=OUT/'feature_cache'/(stem+'.joblib')
    obj=None
    if location.exists():
        try:
            loaded=joblib.load(location)
            if loaded.get('signature')==digest:obj=loaded
        except Exception:pass
    if obj is None:
        obj={'signature':digest,'props':pool,'features':candidate_features(stem,pool,anchors)}
        joblib.dump(obj,location,compress=3)
    if with_labels:
        assert stem in TRAIN and stem not in TEST,'Attempt to read test labels!'
        obj=dict(obj,credits=expert_novelty_credits(stem,obj['props'],anchors))
    return obj

assert len(COLUMNS)==35
print('V29 features:',len(COLUMNS),'| training objective: incremental new-instance IoU credit')


## 9. Train 4 complementary estimators and perform date-grouped OOF selection

The novelty-credit regression target is in [0,1], combining *probability of being an uncovered match* and matched IoU. Three regressors learn credit directly; a classifier learns whether credit is nonzero. Candidate selection is tuned using **OOF predictions on CAL** and full pooled PQ, rather than ROC AUC alone. Different ensemble aggregation rules are evaluated as real methods, not assumed wins.


In [ ]:
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,HistGradientBoostingRegressor,ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score,mean_absolute_error

def make_novelty_models():
    return [
        HistGradientBoostingRegressor(max_iter=180,max_leaf_nodes=9,min_samples_leaf=16,learning_rate=.045,l2_regularization=4.,random_state=SEED+29),
        ExtraTreesRegressor(n_estimators=260,max_depth=12,min_samples_leaf=3,max_features=.77,n_jobs=-1,random_state=SEED+129),
        RandomForestRegressor(n_estimators=220,max_depth=11,min_samples_leaf=4,max_features=.72,n_jobs=-1,random_state=SEED+229),
        ExtraTreesClassifier(n_estimators=250,max_depth=11,min_samples_leaf=4,max_features=.75,class_weight='balanced',n_jobs=-1,random_state=SEED+329)]

def fit_novelty(models,x,y):
    for k,m in enumerate(models):m.fit(x,(y>0).astype(int) if k==3 else y)
    return models

def each_score(models,x):
    if not len(x):return np.zeros((0,4),float)
    values=[np.clip(m.predict(x),0,1) for m in models[:3]]
    values.append(models[3].predict_proba(x)[:,1]*.68)
    return np.stack(values,axis=1)

def combine_scores(a,mode):
    if not len(a):return np.asarray([],float)
    if mode=='mean':return a.mean(axis=1)
    if mode=='pessimistic':return .65*a.mean(axis=1)+.35*np.min(a,axis=1)
    if mode=='rank':
        n=len(a)
        if n<=1:return a.mean(axis=1)
        # Actual rank aggregation; still gate with the absolute predicted PQ credit.
        ranks=np.stack([pd.Series(a[:,j]).rank(method='average',pct=True).to_numpy() for j in range(4)],axis=1)
        return a.mean(axis=1)*(.65+.35*ranks.mean(axis=1))
    if mode=='agreement':return a.mean(axis=1)-.40*a.std(axis=1)
    raise ValueError(mode)

CAL_IDS=sorted(CAL);AUDIT_IDS=sorted(AUDIT)
xs=[];ys=[];groups=[];train_n=[];cal_bank={};CAL_ANCHORS={}
for stem in tqdm(CAL_IDS,desc='Training 4 independent novelty-credit estimators'):
    anchors=frozen_v25_masks(stem);CAL_ANCHORS[stem]=anchors
    info=scene_data(stem,with_labels=True,anchors=anchors)
    x=info['features'];credit=np.asarray(info['credits'],float)
    assert len(x)==len(credit)
    cal_bank[stem]=info;xs.append(x);ys.extend(credit)
    groups.extend([stem[:8]]*len(credit));train_n.append(len(credit))
X=pd.concat(xs,ignore_index=True)[COLUMNS].fillna(0.)
y=np.asarray(ys,float);grp=np.asarray(groups)
assert len(y)>=200 and len(set(grp))>=4, ('Insufficient candidate training data',len(y),len(set(grp)))
print('Novelty credit training:',len(y),'candidates,',int((y>0).sum()),'genuinely novel positives')
NOVELTY_ACTIVE=bool((y>0).sum()>=15 and (y<=0).sum()>=60)
OOF=np.zeros((len(y),4),float)
if NOVELTY_ACTIVE:
    for fold,(ti,vi) in enumerate(GroupKFold(n_splits=4).split(X,(y>0).astype(int),grp)):
        if (y[ti]>0).sum()<5 or (y[ti]==0).sum()<20:
            NOVELTY_ACTIVE=False;break
        learner=fit_novelty(make_novelty_models(),X.iloc[ti],y[ti])
        OOF[vi]=each_score(learner,X.iloc[vi]);print('Date-group OOF',fold+1,len(vi))
if NOVELTY_ACTIVE:
    NOVELTY_MODELS=fit_novelty(make_novelty_models(),X,y)
    joblib.dump({'models':NOVELTY_MODELS,'columns':COLUMNS,'train_days':sorted(set(grp)),
        'credits_are_conditional_on_V25':True,'test_labels_used':False},
        '/kaggle/working/SOLAR_V29_NOVELTY_RANKERS.joblib')
else:
    NOVELTY_MODELS=None;print('Insufficient unique-positive evidence; new candidates disabled.')
# Predictor receives a different set of candidates per image, so compute rank-based
# scores scene by scene even for OOF. Do not normalize across the entire dataset.
CAL_OOF={};offset=0
for stem,n in zip(CAL_IDS,train_n):
    CAL_OOF[stem]=OOF[offset:offset+n];offset+=n
assert offset==len(OOF)

def candidate_policy(anchors,proposals,scores,threshold,max_add=MAX_NEW_ACCEPTED_PER_SCENE,**_):
    selected=[];all_masks=list(anchors)
    occupied=np.zeros((H,W),np.uint8)
    for rr in anchors:occupied|=coco.decode(rr).astype(np.uint8)
    ordering=np.argsort(-np.asarray(scores,float)) if len(scores) else []
    for index in ordering:
        if len(selected)>=max_add:break
        score=float(scores[index])
        if score<threshold:break
        rr=as_rle(proposals[index]['counts'])
        if overlap_max(rr,all_masks)>=MAX_ACCEPTED_OVERLAP:continue
        pix=coco.decode(rr).astype(np.uint8)
        clean=pix&(occupied==0)
        if int(clean.sum())<MIN_EXTRA_AREA or clean.sum()/max(1,pix.sum())<MIN_FREE_FRACTION:continue
        rec=enc(clean);selected.append(rec);all_masks.append(rec);occupied|=clean
    return list(anchors)+selected

METHOD_MODES=('mean','rank','pessimistic','agreement')
GRID={}
BASE_CAL=np.zeros(4,float)
for stem in CAL_IDS:BASE_CAL+=physical_counts(stem,CAL_ANCHORS[stem])
BASELINE_CAL=pq(BASE_CAL)
if NOVELTY_ACTIVE:
    for mode in METHOD_MODES:
        for threshold in CANDIDATE_THRESHOLDS:
            pooled=np.zeros(4,float)
            for stem in CAL_IDS:
                anchors=CAL_ANCHORS[stem];info=cal_bank[stem]
                ranking=combine_scores(CAL_OOF[stem],mode)
                pooled+=physical_counts(stem,candidate_policy(anchors,info['props'],ranking,threshold))
            GRID[(mode,threshold)]=pq(pooled)
else:
    for mode in METHOD_MODES:
        for th in CANDIDATE_THRESHOLDS:GRID[(mode,th)]=BASELINE_CAL
BEST_THRESHOLDS={mode:max(CANDIDATE_THRESHOLDS,key=lambda th:(GRID[(mode,th)],th)) for mode in METHOD_MODES}
print('CAL OOF V25 baseline',round(BASELINE_CAL,5),'chosen thresholds:',BEST_THRESHOLDS)
for mode in METHOD_MODES:
    print(mode,'OOF PQ',round(GRID[(mode,BEST_THRESHOLDS[mode])],5))
report={'training_candidates':len(y),'positive_credits':int((y>0).sum()),'active':NOVELTY_ACTIVE,
        'CAL_V25_PQ':BASELINE_CAL,'OOF_PQ':{str(k):v for k,v in GRID.items()},
        'thresholds':BEST_THRESHOLDS,'train_dates':len(set(grp)),
        'warning':'Previously used V20 holdout. Only train labels used; no independent validation claim.'}
save_json(OUT/'reports'/'V29_novelty_training.json',report)


## 9B. Competitor-inspired direct rank averaging of V24 and V25 instance-quality models

Shelamanov's reported 0.40 method highlights rank averaging. We test this directly on the **existing detector proposals** (not only on novel candidates). V24's 15-feature quality model and V25's 69-feature two-model ensemble provide distinct probability estimates; mix their calibrated scores or per-scene ranks and re-select the already-valid detector proposals. This is measured against V25 without retraining the detectors and without touching test labels.


In [ ]:
DIRECT_RANK_MODES=('blend','rankavg')
DIRECT_RANK_THRESHOLDS=[.20,.24,.28,.32,.36,.40,.44,.48,.52]
DIRECT_RANK_MEMORY={}

def direct_rank_data(stem):
    if stem in DIRECT_RANK_MEMORY:return DIRECT_RANK_MEMORY[stem]
    info=scene_feature_data(stem)
    n=int(info['n_base']);base=[as_rle(v) for v in info['masks'][:n]]
    if n==0:
        out={'base':[],'v25':np.asarray([],float),'v24':np.asarray([],float),'refined':[]}
    else:
        frame=info['features'].iloc[:n]
        q25=rank_probs(new_rankers,frame)
        q24=quality_v24.predict_proba(frame[V24_FEATURES].fillna(0.))[:,1]
        gray=cv2.imread(str(TRAIN.get(stem) or TEST.get(stem)),0)
        masks=[real_refine(stem,rr,gray) if q25[i]>=.50 else rr for i,rr in enumerate(base)]
        out={'base':base,'v25':q25,'v24':q24,'refined':masks}
    DIRECT_RANK_MEMORY[stem]=out
    return out

def direct_rank_masks(stem,kind,threshold):
    d=direct_rank_data(stem)
    q25=d['v25'];q24=d['v24']
    if not len(q25):return []
    if kind=='blend':q=.60*q25+.40*q24
    elif kind=='rankavg':
        if len(q25)==1:q=.5*(q25+q24)
        else:
            r25=pd.Series(q25).rank(pct=True).to_numpy()
            r24=pd.Series(q24).rank(pct=True).to_numpy()
            q=.70*(.5*(q25+q24))+.30*(.5*(r25+r24))
    else:raise ValueError(kind)
    return nonoverlap_after_refinement([m for m,score in zip(d['refined'],q) if score>=threshold])

DIRECT_THRESHOLDS={};DIRECT_CAL_PQ={}
for kind in DIRECT_RANK_MODES:
    for th in DIRECT_RANK_THRESHOLDS:
        vals=np.zeros(4,float)
        for stem in CAL_IDS:vals+=physical_counts(stem,direct_rank_masks(stem,kind,th))
        DIRECT_CAL_PQ[(kind,th)]=pq(vals)
    DIRECT_THRESHOLDS[kind]=max(DIRECT_RANK_THRESHOLDS,
        key=lambda th:(DIRECT_CAL_PQ[(kind,th)],th))
print('Direct V24/V25 rank fusion chosen calibration thresholds:',DIRECT_THRESHOLDS)
for kind in DIRECT_RANK_MODES:
    print('Direct',kind,'CAL PQ',round(DIRECT_CAL_PQ[(kind,DIRECT_THRESHOLDS[kind])],5),
          'vs original V25',round(BASELINE_CAL,5))
save_json(OUT/'reports'/'V29_direct_rank_training.json',{
    'methods':list(DIRECT_RANK_MODES),'cutoffs':DIRECT_THRESHOLDS,
    'CAL_PQ':{str(k):v for k,v in DIRECT_CAL_PQ.items()},
    'historical_holdout_reused':True,'audit_not_used_for_thresholds':True})


## 10. Independently scored ablations and day-paired bootstrap

Audit dates were not used for training or tuning thresholds in V29. Compare PQ, TP, FP, FN and day-paired uncertainty against the 0.39 V25 reconstruction. Any attempt to add candidates that does not **credibly** improve audit PQ is an experimental CSV only; `submission.csv` retains the original byte-verified winner. This explicitly prevents score deterioration from more candidate predictions.


In [ ]:
METHODS=['V25_PROVEN_039']+['V29_'+m.upper()+'_RANK' for m in METHOD_MODES]+['V29_V24V25_BLEND','V29_V24V25_RANKAVG']
METHOD_TO_MODE=dict(zip(['V29_'+m.upper()+'_RANK' for m in METHOD_MODES],METHOD_MODES))

def method_masks(stem,name,anchors=None,precomputed=None):
    anchors=list(anchors) if anchors is not None else frozen_v25_masks(stem)
    if name=='V25_PROVEN_039':return anchors
    if name=='V29_V24V25_BLEND':return direct_rank_masks(stem,'blend',DIRECT_THRESHOLDS['blend'])
    if name=='V29_V24V25_RANKAVG':return direct_rank_masks(stem,'rankavg',DIRECT_THRESHOLDS['rankavg'])
    if not NOVELTY_ACTIVE:return anchors
    info=precomputed if precomputed is not None else scene_data(stem,anchors=anchors)
    if not len(info['features']):return anchors
    scores=each_score(NOVELTY_MODELS,info['features'][COLUMNS])
    mode=METHOD_TO_MODE[name]
    quality=combine_scores(scores,mode)
    return candidate_policy(anchors,info['props'],quality,BEST_THRESHOLDS[mode])

AUDIT_COUNTS={name:np.zeros(4,float) for name in METHODS};audit_records=[]
for stem in tqdm(AUDIT_IDS,desc='V29 separated-acquisition-day PQ audit'):
    anchors=frozen_v25_masks(stem)
    feature_obj=scene_data(stem,anchors=anchors)
    for name in METHODS:
        pred=method_masks(stem,name,anchors,feature_obj)
        stats=physical_counts(stem,pred)
        AUDIT_COUNTS[name]+=stats
        audit_records.append({'stem':stem,'day':stem[:8],'method':name,
            'sum_iou':stats[0],'TP':stats[1],'FP':stats[2],'FN':stats[3],
            'instances':len(pred)})
reference=pq(AUDIT_COUNTS['V25_PROVEN_039'])
leader=pd.DataFrame([{'method':name,**metrics(val),'delta_vs_V25':pq(val)-reference}
   for name,val in AUDIT_COUNTS.items()]).sort_values('PQ',ascending=False)
leader.to_csv(OUT/'reports'/'V29_AUDIT_PQ.csv',index=False)
pd.DataFrame(audit_records).to_csv(OUT/'reports'/'V29_AUDIT_PER_SCENE.csv',index=False)
display(leader)
assert abs(reference-.458880)<.033,('Unexpected reconstructed V25 reference',reference)
by_day_agg=pd.DataFrame(audit_records).groupby(['method','day'])[['sum_iou','TP','FP','FN']].sum()
audit_days=sorted(set(z['day'] for z in audit_records))
mat={m:np.stack([by_day_agg.loc[(m,day)].to_numpy(float) for day in audit_days]) for m in METHODS}
bootids=np.random.default_rng(SEED+29).integers(0,len(audit_days),size=(1200,len(audit_days)))
def batch_pq(a):
    den=a[:,1]+.5*(a[:,2]+a[:,3])
    return np.divide(a[:,0],den,out=np.zeros(len(a)),where=den>0)
refboot=batch_pq(mat['V25_PROVEN_039'][bootids].sum(1))
bootstrap=[]
for m in METHODS[1:]:
    delta=batch_pq(mat[m][bootids].sum(1))-refboot
    bootstrap.append({'method':m,'audit_delta':pq(AUDIT_COUNTS[m])-reference,
      'lower_95':np.quantile(delta,.025),'upper_95':np.quantile(delta,.975),
      'fraction_positive':float((delta>0).mean())})
boot_report=pd.DataFrame(bootstrap).sort_values('audit_delta',ascending=False)
boot_report.to_csv(OUT/'reports'/'V29_PAIRED_DAY_BOOTSTRAP.csv',index=False)
display(boot_report)
passing=boot_report[(boot_report.audit_delta>=MIN_PROMOTION_AUDIT_GAIN)&
                    (boot_report.fraction_positive>=MIN_BOOTSTRAP_POSITIVE)]
SELECTED=str(passing.iloc[0].method) if len(passing) else 'V25_PROVEN_039'
print('Default selected after independent-date audit:',SELECTED)
save_json(OUT/'reports'/'V29_selection.json',{'method':SELECTED,'historical_public_PQ':.39,
   'new_test_public_PQ':None,'novelty_active':NOVELTY_ACTIVE,
   'audit_PQ':{k:pq(v) for k,v in AUDIT_COUNTS.items()},'test_labels_read':False,
   'V20_holdout_reused':True,'evaluator_note':'Approximation to official PQ; verify organizer harness.'})


## 11. All 180 test scenes, correctly labeled CSVs, and exact V25 fallback

Unlike V26–V28, novel candidates on test images are associated with masks loaded from the **exact previously scored 0.39 CSV**, not from an imperfect regeneration. This keeps existing instances byte-consistent within experimental fusions. The original CSV is physically copied to `submission.csv` when the audit gate does not pass.


In [ ]:
from contextlib import ExitStack
TEST_STEMS=sorted(TEST);assert len(TEST_STEMS)==180
subpaths={m:OUT/'submissions'/(m+'.csv') for m in METHODS}
row_log=[]
with ExitStack() as stack:
    writers={m:csv.writer(stack.enter_context(open(path,'w',newline='',encoding='utf8'))) for m,path in subpaths.items()}
    for w in writers.values():w.writerow(['filament_id','segmentation_rle'])
    for stem in tqdm(TEST_STEMS,desc='Generate V29 test variants from proven V25 anchors'):
        old=PROVEN_BY_STEM.get(stem,[])
        # No test masks, labels or benchmark metadata are accessed.
        feature_obj=scene_data(stem,anchors=old) if NOVELTY_ACTIVE else None
        for name in METHODS:
            predicted=method_masks(stem,name,anchors=old,precomputed=feature_obj)
            row_log.append({'stem':stem,'method':name,'count':len(predicted)})
            for k,rr in enumerate(predicted,1):
                writers[name].writerow([f'{stem}_{k}',txt(rr)])
        DIRECT_RANK_MEMORY.pop(stem,None)
counts=pd.DataFrame(row_log);counts.to_csv(OUT/'reports'/'V29_TEST_COUNTS.csv',index=False)
manifest={}
for name,path in subpaths.items():
    frame=pd.read_csv(path,dtype=str)
    assert frame.columns.tolist()==['filament_id','segmentation_rle']
    assert frame.filament_id.is_unique and frame.segmentation_rle.notna().all()
    assert frame.filament_id.str.rsplit('_',n=1).str[0].isin(TEST).all()
    assert counts[counts.method==name].stem.nunique()==180
    assert len(frame)==int(counts[counts.method==name]['count'].sum())
    for rle in frame.segmentation_rle:assert int(coco.area(as_rle(rle)))>0
    manifest[name]={'rows':len(frame),'sha256':sha(path),'path':str(path)}
    print(name,len(frame),'SHA',manifest[name]['sha256'][:20])
# Regenerated baseline might have a different CSV quoting or row order. The control
# is the original file's hash, not the regenerated one.
MAIN=Path('/kaggle/working/submission.csv')
shutil.copy2(PROVEN if SELECTED=='V25_PROVEN_039' else subpaths[SELECTED],MAIN)
for i,m in enumerate(METHODS[1:],1):shutil.copy2(subpaths[m],Path('/kaggle/working')/f'V29_ALT_{i}.csv')
assert sha(PROVEN).startswith(V25_EXPECTED_CSV_SHA)
if SELECTED=='V25_PROVEN_039':assert sha(MAIN)==sha(PROVEN)
save_json(OUT/'reports'/'V29_manifest.json',{'public_confirmed':.39,'test_public_new':None,
    'control_sha256':sha(PROVEN),'default':SELECTED,'candidate_files':manifest,
    'test_annotations_used':False,'new_estimator_training':NOVELTY_ACTIVE})
print('FINAL',MAIN,'Selected',SELECTED,'| exact 0.39 control at',PROVEN)


## 12. Archive trained rankers, candidate CSVs, and a reproducible competition report

Keep the report; the organizers' judging rubric includes morphology and technical reproducibility, not just leaderboard PQ. Notes explain limitations: old validation reused, model-derived estimates uncertain, no external ground-truth/test labels, and candidate quality measured by official instance matching only when audited with the organizer's implementation.


In [ ]:
REPORT_ZIP=Path('/kaggle/working/SOLAR_V29_PREDICTIONS_REPORTS.zip')
report_md=f"""# V29 Competition Method Report\n\n**Historical public benchmark**: 0.39 PQ (original V25 CSV SHA {sha(PROVEN)}).\n\n**Hypothesis**: Conditional instance-novelty credit improves ranking of suppressed detector proposals and plausible fragment unions.\n\n**Dataset**: official MAGFiLO competition training images only; 603 earlier train scenes for pretrained V20 detectors; 104 historical validation scenes split into CAL({len(CAL)}) and AUDIT({len(AUDIT)}) by date. CAL was explored in earlier experiments; this audit is *not* independent external evaluation.\n\n**Inference**: global YOLO11m 1792 + native tile YOLO11m 1280, trained V24/V25 ranking and instance refiner, lower-confidence discarded proposals, V25 curve nominations, and optional two-fragment masks; incremental-credited rank averages and pooled-PQ-tuned instance selection.\n\n**New model**: four complementary learned models: histogram gradient boosting regressor, extra trees regressor, random forest regressor, extra trees binary classifier. Labels are per-expert *newly recovered* matched IoU credit (if IoU>0.5 and not previously matched by V25).\n\n**Control**: SHA-protected original scored file; never reconstructed as an alleged exact historical file.\n\n**Automatic selection**: {SELECTED}; public PQ of the new candidate is UNKNOWN.\n\n**Leakage handling**: test images used only for inference; no external MAGFiLO test annotations or file lookup; no leaderboard-driven use of identity labels.\n\n**Prior work**: Artem Shelamanov, solar portfolio (reported 0.40 from rank averaging); organizers' Kaggle 2026 competition announcements and self-evaluation notebook; HeShen et al. public detect/refine ablations; bcap52 public multi-annotator 5-fold and boundary-weighted network.\n"""
(OUT/'reports'/'V29_TECHNICAL_REPORT.md').write_text(report_md)
with zipfile.ZipFile(REPORT_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3,allowZip64=True) as z:
    for file in sorted((OUT/'reports').glob('V29_*')):z.write(file,'reports/'+file.name)
    for file in sorted((OUT/'submissions').glob('*.csv')):z.write(file,'submissions/'+file.name)
    z.write(MAIN,'submission.csv');z.write(PROVEN,PROVEN.name)
with zipfile.ZipFile(REPORT_ZIP) as z:assert z.testzip() is None
MODEL_ZIP=Path('/kaggle/working/SOLAR_V29_MODEL_AND_REPORTS.zip')
with zipfile.ZipFile(MODEL_ZIP,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    for model_path in WEIGHTS.values():z.write(model_path,'models/'+model_path.name)
    for model_path in (V24_REFINER_PATH,V24_QUALITY_PATH,V25_REAL_PATH,V25_CURVE_PATH,V25_QUALITY_PATH):
        z.write(model_path,'models/'+model_path.name)
    if NOVELTY_ACTIVE:z.write('/kaggle/working/SOLAR_V29_NOVELTY_RANKERS.joblib','models/SOLAR_V29_NOVELTY_RANKERS.joblib')
    z.write(REPORT_ZIP,REPORT_ZIP.name)
with zipfile.ZipFile(MODEL_ZIP) as z:assert z.testzip() is None
print('Finished; verified earlier 0.39, new public PQ not yet tested.')
print('Default:',MAIN)
print('All CSVs/reports:',REPORT_ZIP,round(REPORT_ZIP.stat().st_size/2**20,2),'MiB')
print('Full reusable checkpoint bundle:',MODEL_ZIP,round(MODEL_ZIP.stat().st_size/2**20,2),'MiB')


## Read these before submitting

1. `solar_v29/reports/V29_AUDIT_PQ.csv` measures real instance matching against the known V25 reconstruction; `V29_PAIRED_DAY_BOOTSTRAP.csv` shows uncertainty.
2. `solar_v29/reports/V29_novelty_training.json` reveals how many previously missed **unique** instances were learnable, and whether model evidence was sufficient.
3. `V29_ALT_1.csv`–`V29_ALT_6.csv` represent different ranking and risk aggregation methods. These are not equivalent to a new public best without an actual Kaggle score.
4. `submission.csv` is either the exact proven 0.39 V25 file or an automatically selected new method that crossed the audit gate.
5. Preserve `SOLAR_V29_MODEL_AND_REPORTS.zip`. Do not use publicly exposed test annotations, metadata matching, leaderboard exploitation or external labeled competition test cases.

**Scientific note:** The reduction of PQ to `2 * sum(matched IoU) / (n_pred + n_gt)` is algebraically exact under one-to-one matching. This is the motivation for estimating credit for previously missed objects and penalizing every added prediction. It does not make calibrated credit or validation transfer perfect.
